# Bring your own CSV: a synthetic walkthrough

This example creates artificial text, imports a CSV and runs evaluation. Replace only the input path with your own authorized, pre-cleaned dataset. The importer splits on whitespace; it does not extract speakers or clean raw clinical transcripts. All tasks for one participant must share a participant ID and label.

In [ ]:
from pathlib import Path
import tempfile
import csv
from IPython.display import Image, display
from cogtask_ci import import_csv, validate_dataset, evaluate, Config
from cogtask_ci.showcase import scenario_rows
work = Path(tempfile.mkdtemp(prefix="cogtask_ci_csv_"))
source = work / "synthetic_text.csv"
records = scenario_rows("complementary", subjects=48)
# Deliberately omit one task record to demonstrate coverage reporting.
records = records[1:]
with source.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["subject_id", "task", "label", "text"])
    writer.writeheader()
    for row in records:
        writer.writerow({"subject_id": row["subject_id"], "task": row["task"], "label": row["label"],
                         "text": " ".join(" ".join(s) for s in row["segments"])})
print(source)
print(source.read_text(encoding="utf-8").splitlines()[:3])

In [ ]:
dataset = work / "synthetic_text.jsonl"
coverage = import_csv(source, dataset)
assert coverage["subjects"] == 48
assert coverage["complete_subjects"] == 47
assert coverage["incomplete_subjects"] == 1
validate_dataset(dataset)

The incomplete participant can contribute available training records, while every task budget uses the same 47-person complete-task evaluation set. These numbers describe this artificial example, not a study cohort. If your real data cannot support every required inner/outer class split, evaluation stops; do not repeatedly adjust settings to chase a better result.

In [ ]:
metrics = evaluate(dataset, work / "run", Config(outer_folds=3, inner_folds=2, bootstrap=50))
display(Image(filename=str(work / "run" / "task_budget.png")))
print("Report:", work / "run" / "report.html")